# ARIMA：时间序列预测的经典模型
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：10_时间序列 | 源文件：ARIMA.py | 核心功能：平稳性检验、差分、自相关分析、statsmodels 实现

## 概述

ARIMA（AutoRegressive Integrated Moving Average）是时间序列预测的经典统计模型。三个参数：p（自回归阶数）、d（差分阶数）、q（移动平均阶数）。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
# --- 导入库 ---
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 数学原理

### 1. ARIMA(p, d, q) 模型

ARIMA 由三部分组成：自回归（AR）、差分（I）、滑动平均（MA）。

**AR(p) 自回归**：

$$y_t = c + \sum_{i=1}^{p} \phi_i y_{t-i} + \epsilon_t$$

当前值 $y_t$ 是过去 $p$ 个值的线性组合加白噪声。

**MA(q) 滑动平均**：

$$y_t = \mu + \epsilon_t + \sum_{j=1}^{q} \theta_j \epsilon_{t-j}$$

当前值 $y_t$ 是过去 $q$ 个误差的线性组合。

**ARMA(p, q)**（平稳序列）：

$$y_t = c + \sum_{i=1}^{p}\phi_i y_{t-i} + \epsilon_t + \sum_{j=1}^{q}\theta_j \epsilon_{t-j}$$

### 2. 差分与平稳性

**一阶差分**：$\nabla y_t = y_t - y_{t-1}$

**$d$ 阶差分**：$\nabla^d y_t = \nabla(\nabla^{d-1} y_t)$

差分消除趋势，使序列平稳。ARIMA 对差分后的序列 $\nabla^d y_t$ 拟合 ARMA(p, q)。

### 3. ADF 平稳性检验

原假设 $H_0$：序列存在单位根（非平稳）。检验统计量：

$$\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{j=1}^{k}\delta_j \Delta y_{t-j} + \epsilon_t$$

检验 $\gamma = 0$ 是否成立。$p < 0.05$ 时拒绝 $H_0$，认为序列平稳。

### 4. ACF 与 PACF 定阶

| 模型 | ACF | PACF |
|------|-----|------|
| AR(p) | 拖尾（指数衰减） | $p$ 阶截尾 |
| MA(q) | $q$ 阶截尾 | 拖尾 |
| ARMA(p,q) | 拖尾 | 拖尾 |

**截尾**：在某阶之后突然变为 0（统计显著范围内）。**拖尾**：逐渐衰减。

### 5. 模型选择准则

**AIC**（Akaike 信息准则）：

$$\text{AIC} = -2\ln \hat{L} + 2k$$

**BIC**（贝叶斯信息准则）：

$$\text{BIC} = -2\ln \hat{L} + k\ln n$$

其中 $\hat{L}$ 是最大似然值，$k$ 是参数个数。AIC/BIC 越小越好。

### 6. 预测

ARIMA 的 $h$ 步预测：

$$\hat{y}_{T+h|T} = c + \sum_{i=1}^{p}\phi_i \hat{y}_{T+h-i|T} + \sum_{j=1}^{q}\theta_j \hat{\epsilon}_{T+h-j}$$

其中 $\hat{y}_{T+h-i|T} = y_{T+h-i}$（当 $h-i \leq 0$），预测区间随 $h$ 增大而变宽。

### 7. Ljung-Box 检验

检验残差是否为白噪声（模型充分）：

$$Q(k) = n(n+2)\sum_{j=1}^{k}\frac{\hat{\rho}_j^2}{n-j}$$

$H_0$：残差无自相关。$p > 0.05$ 表示模型拟合充分。

### 8. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `adfuller(序列)` | ADF 检验 $\gamma=0$ |
| `ARIMA(序列, order=(p,d,q))` | ARIMA(p,d,q) 模型 |
| `acf(序列, nlags=40)` | 计算样本自相关函数 $\hat{\rho}_k$ |
| `pacf(序列, nlags=40)` | 计算样本偏自相关函数 $\hat{\phi}_{kk}$ |
| `acorr_ljungbox(resid)` | Ljung-Box 白噪声检验 |
| `mean_squared_error` | $\text{MSE} = \frac{1}{n}\sum(y_i - \hat{y}_i)^2$ |
| `forecast(steps=20)` | $h$ 步预测 $\hat{y}_{T+h|T}$ |

### 1. 生成合成时间序列

运行 1. 生成合成时间序列 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
n = 200

# 趋势 + 季节性 + 噪声
t = np.arange(n)
趋势 = 0.05 * t
季节性 = 2 * np.sin(2 * np.pi * t / 30)
噪声 = np.random.normal(0, 0.5, n)
数据 = 趋势 + 季节性 + 噪声

序列 = pd.Series(数据, index=pd.date_range("2020-01-01", periods=n, freq="D"))
print("=" * 50)
print("原始时间序列统计量：")
print(f"  样本数: {n}")
print(f"  均值: {序列.mean():.4f}")
# --- 输出结果 ---
print(f"  标准差: {序列.std():.4f}")
print("=" * 50)

### 2. 平稳性检验（ADF 检验）

检验时间序列的平稳性，决定差分阶数。

In [ ]:
def adf检验(序列, 名称="序列"):
    """ADF 单位根检验，H0: 序列非平稳"""
    result = adfuller(序列.dropna(), autolag="AIC")
    print(f"\n--- ADF 检验: {名称} ---")
    print(f"  ADF 统计量:  {result[0]:.4f}")
# --- 输出结果 ---
    print(f"  p 值:        {result[1]:.4f}")
    print(f"  滞后阶数:    {result[2]}")
    for k, v in result[4].items():
        print(f"  临界值 ({k}): {v:.4f}")
    平稳 = result[1] < 0.05
# --- 输出结果 ---
    print(f"  结论: {'平稳' if 平稳 else '非平稳'}（显著性水平 0.05）")
    return 平稳

adf检验(序列)

# 差分处理
差分 = 序列.diff().dropna()
print(f"\n一阶差分后：")
adf检验(差分, "一阶差分")

### 3. ACF / PACF 定阶

运行 3. ACF / PACF 定阶 的代码，观察算法在该环节的行为。

In [ ]:
acf值 = acf(差分, nlags=20)
pacf值 = pacf(差分, nlags=20)

print(f"\n--- ACF / PACF（前 10 阶） ---")
print(f"{'阶数':>4}  {'ACF':>8}  {'PACF':>8}")
for i in range(1, 11):
    print(f"  {i:>2}    {acf值[i]:>8.4f}  {pacf值[i]:>8.4f}")

# 简单定阶策略：找显著阶数（超过 95% 置信区间 ≈ 1.96/sqrt(n)）
边界 = 1.96 / np.sqrt(len(差分))
q_阶 = max([i for i in range(1, 11) if abs(acf值[i]) > 边界], default=1)
p_阶 = max([i for i in range(1, 11) if abs(pacf值[i]) > 边界], default=1)
d_阶 = 1  # 已做一阶差分
print(f"\n自动定阶建议: ARIMA({p_阶}, {d_阶}, {q_阶})")

### 4. 模型拟合

运行 4. 模型拟合 的代码，观察算法在该环节的行为。

In [ ]:
train_size = int(n * 0.8)
train, test = 序列[:train_size], 序列[train_size:]

模型 = ARIMA(train, order=(p_阶, d_阶, q_阶))
拟合结果 = 模型.fit()
print(f"\n--- 模型摘要（关键指标） ---")
print(f"  AIC:  {拟合结果.aic:.2f}")
print(f"  BIC:  {拟合结果.bic:.2f}")
# --- 输出结果 ---
print(f"  对数似然: {拟合结果.llf:.2f}")

### 5. 残差诊断

检查模型残差是否满足独立同分布假设。

In [ ]:
残差 = 拟合结果.resid
print(f"\n--- 残差统计 ---")
print(f"  均值:   {残差.mean():.6f}")
print(f"  标准差: {残差.std():.4f}")

# Ljung-Box 检验：H0 残差无自相关
lb_result = acorr_ljungbox(残差.dropna(), lags=[10], return_df=True)
p值 = lb_result["lb_pvalue"].values[0]
print(f"  Ljung-Box 检验 (lag=10): p 值 = {p值:.4f}")
print(f"  残差自相关: {'无显著自相关' if p值 > 0.05 else '存在自相关'}")

### 6. 预测与评估

使用训练好的模型进行预测，观察输出结果。

In [ ]:
forecast = 拟合结果.forecast(steps=len(test))
rmse = np.sqrt(mean_squared_error(test, forecast))
mae = mean_absolute_error(test, forecast)
mape = np.mean(np.abs((test.values - forecast.values) / test.values)) * 100

print(f"\n--- 预测评估 ---")
print(f"  测试集长度: {len(test)}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  MAPE: {mape:.2f}%")

# 展示前 10 个预测值
print(f"\n--- 预测 vs 真实（前 10 个） ---")
print(f"{'日期':>12}  {'真实':>8}  {'预测':>8}  {'误差':>8}")
for i in range(min(10, len(test))):
    实际 = test.values[i]
    预测 = forecast.values[i]
# --- 输出结果 ---
    print(f"  {str(test.index[i].date()):>10}  {实际:>8.3f}  {预测:>8.3f}  {实际 - 预测:>8.3f}")

### 7. 全序列拟合后预测未来

使用训练好的模型进行预测，观察输出结果。

In [ ]:
全模型 = ARIMA(序列, order=(p_阶, d_阶, q_阶))
全拟合 = 全模型.fit()
未来预测 = 全拟合.forecast(steps=30)
print(f"\n--- 未来 30 天预测 ---")
print(f"{'日期':>12}  {'预测值':>8}")
# --- 循环处理 ---
for i in range(30):
    print(f"  {str(未来预测.index[i].date()):>10}  {未来预测.values[i]:>8.3f}")

## 关键代码解释

```python
from statsmodels.tsa.arima.model import ARIMA
model = ARIMA(ts, order=(p, d, q))
fitted = model.fit()
forecast = fitted.forecast(steps=10)
```

## 注意事项

1. 数据必须平稳（均值和方差不随时间变化）
2. ADF 检验判断平稳性
3. ACF/PACF 图辅助选择 p 和 q

## 总结与延伸

以上代码展示了 **ARIMA** 的完整流程。

**解读要点**：
- 观察预测曲线与实际值的 **趋势一致性**
- 关注残差是否呈现随机分布（无明显模式）
- 对比不同模型的 **MAPE / RMSE** 指标

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **SARIMA**：加入季节性
- **ARIMA 的自动选择**：`pmdarima.auto_arima`
- **Prophet**：Facebook 的时间序列工具，更易用
